# 01 — Smoke test: Kumo Tabular on GPU
Checks the environment, installs `structured-data-models`, runs the model-card classification workflow
(`sdm.models.KumoTabular`) on a public dataset, and measures latency and peak GPU memory at 1k/5k/10k context rows
for the Small and Medium sizes. **Runtime → Change runtime type → GPU** before running.

API sources: [model card](https://huggingface.co/nvidia/Kumo-Tabular), [repo README](https://github.com/NVIDIA/structured-data-models),
[API docs](https://nvidia.github.io/structured-data-models/api/generated/sdm.models.KumoTabular).

**Colab setup.** In Google Colab, this clones the repository, moves into it and installs it in editable mode, so
`import kumo_bench` works. Run locally, it changes nothing and only works out the repo root. All output paths
below are built from `ROOT`. Set `REPO_URL` to your GitHub repository before running in Colab.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/<your-username>/Kumo-Benchmark.git"  # TODO: set before first Colab run
if "google.colab" in sys.modules:
    ROOT = Path("/content/Kumo-Benchmark")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("repo root:", ROOT)

**Environment check.** Prints the Python and PyTorch versions and the GPU. The library needs Python ≥ 3.11 and PyTorch ≥ 2.7.

In [ ]:
import sys, torch
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "No CUDA GPU - switch the Colab runtime to GPU"
print("GPU    :", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda)

**Install the library.** Installs from PyPI (the README's documented command).

In [ ]:
%pip install -q structured-data-models

**Pin the installed version.** Records the exact version and writes a lock file at the repo root, so later notebooks
can install it with `pip install -r requirements-lock.txt`.

In [ ]:
from importlib.metadata import version
SDM_VERSION = version("structured-data-models")
with open(ROOT / "requirements-lock.txt", "w") as f:
    f.write(f"structured-data-models=={SDM_VERSION}\n")
print("structured-data-models ==", SDM_VERSION)

**Set and record the random seed.** It seeds Python, NumPy and PyTorch. The same seed also drives the data sampling below.

In [ ]:
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("SEED =", SEED)

**Record the Hugging Face model revision.** The `KumoTabular` constructor takes no revision argument, and the docs say
weights download from `nvidia/Kumo-Tabular` on first use. This cell records the Hub's current commit SHA for that repo. A later cell
checks which revisions actually landed in the local cache.

In [ ]:
from huggingface_hub import HfApi
HF_REPO = "nvidia/Kumo-Tabular"
HF_REVISION = HfApi().model_info(HF_REPO).sha
print(HF_REPO, "@", HF_REVISION)

**Load a public classification dataset.** Forest CoverType (UCI, via scikit-learn): 54 numeric features and 7 classes,
within the model's 10-class single-pass limit. It is large enough for 10k context rows. The cell takes a stratified
pool of 11,000 rows: 10,000 for context and a fixed 1,000-row query set.

In [ ]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
df = fetch_covtype(as_frame=True).frame.rename(columns={"Cover_Type": "target"})
pool, _ = train_test_split(df, train_size=11_000, stratify=df["target"], random_state=SEED)
ctx_df, qry_df = train_test_split(pool, test_size=1_000, stratify=pool["target"], random_state=SEED)
ctx_df, qry_df = ctx_df.reset_index(drop=True), qry_df.reset_index(drop=True)
print(ctx_df.shape, qry_df.shape)

**Convert the data to `TableTensor`s.** This follows the model card: `sdm.infer_stypes` is called with the target
overridden to categorical, then `sdm.TableTensor.from_pandas` builds the tensors on the GPU.

In [ ]:
import sdm
device = torch.device("cuda")
stypes = sdm.infer_stypes(pool, overrides={"target": "categorical"})
ctx = sdm.TableTensor.from_pandas(df=ctx_df, stypes=stypes, device=device)
qry = sdm.TableTensor.from_pandas(df=qry_df, stypes=stypes, device=device)

**Review the inferred semantic types.** Prints the stype that `infer_stypes` assigned to each column. Every feature
should be numerical and `target` categorical. If a column looks wrong, add it to `overrides` above.

In [ ]:
for col, st in (stypes.items() if hasattr(stypes, "items") else zip(pool.columns, stypes)):
    print(f"{col:40s} {st}")

**Majority-class baseline.** Predicts the context set's most frequent class for every query row and scores it on the
same 1,000 query rows. Kumo must beat this number for its results to mean anything.

In [ ]:
majority = ctx_df["target"].mode()[0]
BASELINE_ACC = float((qry_df["target"] == majority).mean())
print(f"majority class = {majority}, baseline accuracy = {BASELINE_ACC:.3f}")

**Inference helper.** Runs the model-card call `model(x_context=, y_context=, x_query=, num_estimators=...)` under fp16
autocast. Timing brackets `torch.cuda.synchronize()` calls, and peak memory is reset before each run.

In [ ]:
import time

def run(model, n_ctx, num_estimators):
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        probs = model(
            x_context=ctx[:n_ctx].drop_columns("target"),
            y_context=ctx[:n_ctx, "target"],
            x_query=qry.drop_columns("target"),
            num_estimators=num_estimators,
        )
    torch.cuda.synchronize()
    return probs, time.perf_counter() - t0, torch.cuda.max_memory_allocated() / 2**20

**Prediction checks.** Asserts that every row of predicted probabilities sums to 1, with a tolerance of 1e-2 to allow for
fp16 rounding. It then scores accuracy and flags any run that fails to beat the majority-class baseline. Mapping
probabilities to labels assumes the output columns follow the sorted class order. That is not documented, so check the
printed shape and accuracy.

In [ ]:
y_true = qry_df["target"].to_numpy()
classes = np.sort(ctx_df["target"].unique())

def check(probs):
    p = torch.as_tensor(probs).float().cpu().numpy()
    assert np.allclose(p.sum(1), 1.0, atol=1e-2), f"probability rows do not sum to 1 (range {p.sum(1).min():.4f}-{p.sum(1).max():.4f})"
    acc = float((classes[p.argmax(1)] == y_true).mean())
    flag = "" if acc > BASELINE_ACC else f"  <-- WARNING: does not beat majority baseline ({BASELINE_ACC:.3f})"
    return p.shape, acc, flag

**Configurations.** Uses `num_estimators=8` (the model-card value) at 1k/5k/10k context rows, and `num_estimators=1` at
1k and 10k only. Each configuration runs for both sizes.

In [ ]:
CONFIGS = [(1_000, 8), (5_000, 8), (10_000, 8), (1_000, 1), (10_000, 1)]

**Benchmark loop.** For each size it loads the model and does one warm-up run (excluded from timing), then times every
configuration on the same 1,000 query rows. If a run hits CUDA out-of-memory, the loop clears the cache, records "OOM",
and moves on to the next configuration.

In [ ]:
rows = []
for size in ["small", "medium"]:
    model = sdm.models.KumoTabular(task="classification", size=size, device=device)
    run(model, 1_000, 8)  # warm-up
    for n_ctx, n_est in CONFIGS:
        row = dict(model=f"Kumo-Tabular-{size}", size=size, context_rows=n_ctx, query_rows=len(qry_df),
                   num_estimators=n_est, baseline_accuracy=BASELINE_ACC)
        try:
            probs, secs, mem = run(model, n_ctx, n_est)
        except torch.cuda.OutOfMemoryError:
            probs = None; torch.cuda.empty_cache()
            row.update(wall_clock_s="OOM", peak_gpu_mem_mib="OOM", query_accuracy="OOM")
            print(f"{size:6s} ctx={n_ctx:>6,} est={n_est}  OOM")
        if probs is not None:
            shape, acc, flag = check(probs)
            row.update(wall_clock_s=secs, peak_gpu_mem_mib=mem, query_accuracy=acc)
            print(f"{size:6s} ctx={n_ctx:>6,} est={n_est}  {secs:7.3f}s  {mem:8.1f} MiB  probs={shape}  acc={acc:.3f}{flag}")
        rows.append(row)
    del model; torch.cuda.empty_cache()

**Verify the cached revisions.** Lists what `huggingface_hub` downloaded for `nvidia/Kumo-Tabular`. If this differs from
`HF_REVISION`, the library pulled another commit, and the cached value is the one to trust.

In [ ]:
from huggingface_hub import scan_cache_dir
cached = sorted({r.commit_hash for repo in scan_cache_dir().repos if repo.repo_id == HF_REPO for r in repo.revisions})
print("cached revisions:", cached or "none found (library may use its own cache)")

**Save the results.** Each row carries the environment, library version, model revision and seed, so the CSV stands on its own.

In [ ]:
import pandas as pd
out = pd.DataFrame(rows).assign(
    python=sys.version.split()[0], torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
    sdm_version=SDM_VERSION, hf_repo=HF_REPO, hf_revision=HF_REVISION,
    hf_cached_revisions=";".join(cached), seed=SEED, dataset="covtype")
(ROOT / "results/raw").mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / "results/raw/smoke_test.csv", index=False)
out